# Plant Disease Detection — Tomato (10 classes)
**Project:** Classify tomato leaf diseases from one photo, using transfer learning.
**Dataset:** PlantVillage (Kaggle *New Plant Diseases Dataset*) — **10 Tomato classes**, split **70% train / 15% validation / 15% test**. Headline result = **test** accuracy.
**Model:** MobileNetV2 (ImageNet) + GAP + Dropout + Dense softmax. Plus: a from-scratch baseline CNN for comparison, and Grad-CAM explainability.

First: Runtime > Change runtime type > **GPU**. Then run cells top to bottom.

In [ ]:
# 1. Setup
import sys
!{sys.executable} -m pip -q install seaborn scikit-learn kagglehub gradio
import tensorflow as tf
print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))

In [ ]:
# 2. Download the dataset (PlantVillage, augmented version on Kaggle)
# kagglehub will ask for / use your Kaggle credentials the first time.
import kagglehub
DATA_ROOT = kagglehub.dataset_download("vipooool/new-plant-diseases-dataset")
print("Dataset at:", DATA_ROOT)

In [ ]:
# 3. Build the 10-class Tomato subset with a 70/15/15 split (seeded, per class)
from pathlib import Path
import os, random, shutil

SELECTED = ["Tomato___healthy", "Tomato___Bacterial_spot", "Tomato___Early_blight",
            "Tomato___Late_blight", "Tomato___Leaf_Mold", "Tomato___Septoria_leaf_spot",
            "Tomato___Spider_mites_Two-spotted_spider_mite", "Tomato___Target_Spot",
            "Tomato___Tomato_mosaic_virus", "Tomato___Tomato_Yellow_Leaf_Curl_Virus"]
SEED = 123
root = Path(DATA_ROOT)
split_root = Path("data_split")
if split_root.exists(): shutil.rmtree(split_root)
rng = random.Random(SEED)
for name in SELECTED:
    cands = [p for p in root.rglob(name) if p.is_dir()]
    assert cands, f"{name} not found under {root}"
    src = sorted(cands, key=lambda p: "train" not in p.parts)[0]  # prefer train/
    files = sorted(p for p in src.iterdir() if p.suffix.lower() in {".jpg", ".jpeg", ".png"})
    rng.shuffle(files)
    n = len(files); n_tr = int(n * 0.70); n_va = int(n * 0.15)
    parts = {"train": files[:n_tr], "val": files[n_tr:n_tr+n_va], "test": files[n_tr+n_va:]}
    for split, fs in parts.items():
        dst = split_root / split / name; dst.mkdir(parents=True, exist_ok=True)
        for img in fs:
            try: os.link(img, dst / img.name)
            except OSError: shutil.copy2(img, dst / img.name)
    print(name, {k: len(v) for k, v in parts.items()})

In [ ]:
# 4. Load the three datasets (224x224)
IMG = (224, 224); BATCH = 32
def load(name, shuffle=False):
    return tf.keras.utils.image_dataset_from_directory(
        split_root / name, image_size=IMG, batch_size=BATCH, shuffle=shuffle, seed=SEED)
train_ds, val_ds, test_ds = load("train", True), load("val"), load("test")
class_names = train_ds.class_names
print(class_names)
# Cache as uint8, not float32: 4x less RAM (~3 GB instead of ~11 GB, which would crash free Colab)
def cache_compact(ds):
    ds = ds.map(lambda x, y: (tf.cast(tf.round(x), tf.uint8), y)).cache()
    return ds.map(lambda x, y: (tf.cast(x, tf.float32), y))
train_ds = cache_compact(train_ds).shuffle(1000).prefetch(tf.data.AUTOTUNE)
val_ds = cache_compact(val_ds).prefetch(tf.data.AUTOTUNE)
test_ds = cache_compact(test_ds).prefetch(tf.data.AUTOTUNE)

In [ ]:
# 5. Class distribution (DA-1 figure) - are the classes balanced?
import matplotlib.pyplot as plt
counts = []
for name in class_names:
    counts.append(len([p for p in (split_root / "train" / name).iterdir()]))
plt.figure(figsize=(11, 3.5))
plt.bar([c.replace("Tomato___", "").replace("_", " ") for c in class_names], counts, color="green")
plt.xticks(rotation=30, ha="right"); plt.ylabel("Train images"); plt.title("Training images per class")
plt.tight_layout(); plt.savefig("class_distribution.png", dpi=150); plt.show()

In [ ]:
# 6. Look at the data: one batch of training images with labels
import matplotlib.pyplot as plt
for images, labels in train_ds.take(1):
    plt.figure(figsize=(9, 9))
    for i in range(9):
        plt.subplot(3, 3, i + 1)
        plt.imshow(images[i].numpy().astype("uint8")); plt.axis("off")
        plt.title(class_names[int(labels[i])].replace("Tomato___", "").replace("_", " "), fontsize=8)
    plt.tight_layout(); plt.savefig("sample_grid.png", dpi=150); plt.show()

## The model
MobileNetV2 already knows edges, textures and shapes from ImageNet. We freeze it, and train only a small head: GlobalAveragePooling -> Dropout(0.2) -> Dense(10, softmax). Then we fine-tune its last 30 layers at a very low learning rate.

In [ ]:
# 7. Build MobileNetV2 transfer-learning model (base frozen)
augment = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.1),
    tf.keras.layers.RandomZoom(0.1)], name="augmentation")
base = tf.keras.applications.MobileNetV2(input_shape=IMG+(3,), include_top=False, weights="imagenet")
base.trainable = False
inputs = tf.keras.Input(shape=IMG+(3,))
x = augment(inputs)
x = tf.keras.applications.mobilenet_v2.preprocess_input(x)
x = base(x, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dropout(0.2)(x)
outputs = tf.keras.layers.Dense(len(class_names), activation="softmax")(x)
model = tf.keras.Model(inputs, outputs)
model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
              loss="sparse_categorical_crossentropy", metrics=["accuracy"])
model.summary()

In [ ]:
# 8. Train - 10 epochs, base frozen
history = model.fit(train_ds, validation_data=val_ds, epochs=10)

In [ ]:
# 9. Fine-tune - unfreeze last 30 layers of MobileNetV2, 5 epochs at lr 1e-5
base.trainable = True
for layer in base.layers[:-30]:
    layer.trainable = False
model.compile(optimizer=tf.keras.optimizers.Adam(1e-5),
              loss="sparse_categorical_crossentropy", metrics=["accuracy"])
history_ft = model.fit(train_ds, validation_data=val_ds, epochs=5)
for k in history.history:
    history.history[k] += history_ft.history[k]
model.save("plant_disease_subset.keras")
import json; json.dump(class_names, open("class_names.json", "w"), indent=2)
print("Model + class_names saved")

In [ ]:
# 10. Training curves - SAVE for the report
import matplotlib.pyplot as plt
acc = history.history["accuracy"]; vacc = history.history["val_accuracy"]
loss = history.history["loss"]; vloss = history.history["val_loss"]
ep = range(1, len(acc) + 1)
fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 4))
a1.plot(ep, acc, label="Train"); a1.plot(ep, vacc, label="Validation")
a1.set_title("Accuracy"); a1.set_xlabel("Epoch"); a1.legend()
a2.plot(ep, loss, label="Train"); a2.plot(ep, vloss, label="Validation")
a2.set_title("Loss"); a2.set_xlabel("Epoch"); a2.legend()
fig.tight_layout(); fig.savefig("training_curves.png", dpi=150); plt.show()

In [ ]:
# 11. Evaluate on the TEST set (never seen in training) - confusion matrix + report
import numpy as np, seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix

def evaluate(m, ds):
    y_true, y_pred = [], []
    for images, labels in ds:
        y_true.extend(labels.numpy())
        y_pred.extend(np.argmax(m.predict(images, verbose=0), axis=1))
    return np.array(y_true), np.array(y_pred)

y_true, y_pred = evaluate(model, test_ds)
test_acc = float(np.mean(y_true == y_pred))
print(f"MobileNetV2 TEST accuracy: {test_acc*100:.2f}%")
print(classification_report(y_true, y_pred, target_names=class_names))
short = [c.replace("Tomato___", "").replace("_", " ") for c in class_names]
plt.figure(figsize=(10, 8))
sns.heatmap(confusion_matrix(y_true, y_pred), annot=True, fmt="d", cmap="Greens",
            xticklabels=short, yticklabels=short)
plt.xlabel("Predicted"); plt.ylabel("Actual"); plt.title("Confusion Matrix (test set)")
plt.tight_layout(); plt.savefig("confusion_matrix_test.png", dpi=150); plt.show()

## Baseline experiment
Same data, same split — but a small CNN trained **from scratch** (no ImageNet weights). The comparison table is the experiment: it shows what transfer learning actually buys you.

In [ ]:
# 12. Baseline CNN from scratch - build + train (10 epochs)
baseline = tf.keras.Sequential([
    tf.keras.layers.Rescaling(1./255, input_shape=IMG+(3,)),
    tf.keras.layers.Conv2D(32, 3, activation="relu"),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Conv2D(64, 3, activation="relu"),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Conv2D(128, 3, activation="relu"),
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(len(class_names), activation="softmax"),
], name="baseline_cnn")
baseline.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
                 loss="sparse_categorical_crossentropy", metrics=["accuracy"])
baseline.fit(train_ds, validation_data=val_ds, epochs=10)
b_true, b_pred = evaluate(baseline, test_ds)
baseline_acc = float(np.mean(b_true == b_pred))
print(f"Baseline TEST accuracy: {baseline_acc*100:.2f}%")

In [ ]:
# 13. Comparison table - the DA-3 result
import pandas as pd
table = pd.DataFrame([
    {"Model": "Baseline CNN (from scratch)", "Params": baseline.count_params(), "Test accuracy": baseline_acc},
    {"Model": "MobileNetV2 (transfer learning)", "Params": model.count_params(), "Test accuracy": test_acc},
])
print(table.to_string(index=False))
print(f"Transfer learning gain: {(test_acc - baseline_acc)*100:+.2f} points")

## Grad-CAM (class activation map)
Our head is GAP -> Dense, so the activation map is exact: take the last conv feature maps (7x7x1280), weight them by the Dense layer's weights for the predicted class, ReLU, upsample, overlay. If the heat is on the disease spots — not the background — the model is looking at the right thing. Try it on a **wrong** prediction too (check the confusion matrix for a confused pair).

In [ ]:
# 14. Grad-CAM on one test image
import cv2
base_layer = next(l for l in model.layers if isinstance(l, tf.keras.Model) and "mobilenet" in l.name)
dense_layer = next(l for l in model.layers if isinstance(l, tf.keras.layers.Dense))

sample = next(p for p in Path("data_split/test").rglob("*") if p.suffix.lower() in {".jpg", ".jpeg", ".png"})
print("Sample:", sample)
img = tf.keras.utils.load_img(sample, target_size=IMG)
arr = np.expand_dims(tf.keras.utils.img_to_array(img), 0)  # raw 0-255: the model preprocesses internally
probs = model.predict(arr, verbose=0)[0]
pred = int(np.argmax(probs))
print("Prediction:", class_names[pred], f"{probs[pred]*100:.2f}%")

# preprocess -> MobileNetV2 base, as inside the model (Keras 3 can't slice a nested model's output)
feats = base_layer(tf.keras.applications.mobilenet_v2.preprocess_input(tf.constant(arr)), training=False)[0]  # (7, 7, 1280)
cam = np.maximum(feats.numpy() @ dense_layer.kernel[:, pred].numpy(), 0)  # ReLU
cam = cam / (cam.max() + 1e-8)
heat = cv2.applyColorMap(np.uint8(255 * cv2.resize(cam, IMG)), cv2.COLORMAP_JET)
orig = cv2.cvtColor(np.array(img), cv2.COLOR_RGB2BGR)
overlay = cv2.addWeighted(orig, 0.6, heat, 0.4, 0)
plt.figure(figsize=(9, 3.5))
for i, (im, t) in enumerate([(np.array(img), "Original"),
                             (cv2.cvtColor(heat, cv2.COLOR_BGR2RGB), "Heatmap"),
                             (cv2.cvtColor(overlay, cv2.COLOR_BGR2RGB), "Overlay")]):
    plt.subplot(1, 3, i + 1); plt.imshow(im); plt.title(t); plt.axis("off")
plt.suptitle(class_names[pred].replace("Tomato___", "").replace("_", " "))
plt.tight_layout(); plt.savefig("gradcam_sample.png", dpi=150); plt.show()

In [ ]:
# 15. Predict any single image (change the path - try a test image or upload your own)
def predict_image(path):
    im = tf.keras.utils.load_img(path, target_size=IMG)
    a = np.expand_dims(tf.keras.utils.img_to_array(im), 0)  # raw 0-255: the model preprocesses internally
    p = model.predict(a, verbose=0)[0]
    for i in np.argsort(p)[::-1][:3]:
        label = class_names[i].replace("Tomato___", "Tomato - ").replace("_", " ")
        print(f"{label:30s} {p[i]*100:6.2f}%")
predict_image(sample)

## Demo app + what to submit
- **Demo:** run `app.py` from the repo (Gradio). In Colab: `!pip -q install gradio` then `%run app.py` — it prints a share link. Upload a leaf, get disease + confidence + an info card (cause / symptoms / prevention).
- **DA-1 figures:** class distribution (cell 5), sample grid (cell 6)
- **DA-2:** this notebook + training curves (cell 10)
- **DA-3:** test results + confusion matrix (cell 11), comparison table (cell 13), Grad-CAM (cell 14), demo screenshots
- **Expected:** test accuracy usually lands in the 90s on this clean dataset. Report YOUR number, and state the plain-background limitation honestly.